# HAVOC shoes-model pretrain (RunPod)

Orchestrates: env setup -> data prep -> pretrain -> monitor.

**Assumptions**
- Repo cloned at `/workspace/havoc`
- Raw data uploaded to `/workspace/havoc_data/raw` with subfolders
  `smollm/fineweb-edu-dedup/`, `smollm/cosmopedia-v2/`, `tinystories/`, `oasst2/`
- Single CUDA GPU (RTX 4090 or comparable)

In [ ]:
# Cell 1 ── Environment setup
import os, subprocess, sys

REPO = '/workspace/havoc'
os.chdir(REPO)

os.environ.setdefault('HF_HOME',          '/workspace/.cache/huggingface')
os.environ.setdefault('HF_DATASETS_CACHE','/workspace/.cache/huggingface/datasets')
os.environ.setdefault('TRANSFORMERS_CACHE','/workspace/.cache/huggingface/hub')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

# PyTorch ships with RunPod images; install the rest.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                       '-r', 'requirements.txt'])

import torch
print('python  :', sys.version.split()[0])
print('torch   :', torch.__version__, ' cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu     :', torch.cuda.get_device_name(0))

In [ ]:
# Cell 2 ── Data preparation
# Reads C:\havoc_data layout (on RunPod: /workspace/havoc_data/raw),
# tokenizes with GPT-2 BPE, writes uint16 shards to data/shards/.
import subprocess, sys

RAW_DIR = '/workspace/havoc_data/raw'
OUT_DIR = '/workspace/havoc/data/shards'

cmd = [
    sys.executable, 'data/prepare_pretrain.py',
    '--raw_dir',  RAW_DIR,
    '--out_dir',  OUT_DIR,
    '--budget_fineweb_edu',   str(1_800_000_000),
    '--budget_cosmopedia_v2', str(400_000_000),
    '--budget_tinystories',   str(300_000_000),
    '--budget_oasst2',        str(200_000_000),
    '--val_tokens',           str(2_000_000),
    '--tokens_per_shard',     str(100_000_000),
]
print(' '.join(cmd))
subprocess.check_call(cmd)

In [ ]:
# Cell 3 ── Launch pretraining
# Single-GPU. Resume by setting RESUME = 'checkpoints/latest.pt'.
import subprocess, sys

RESUME = None  # or 'checkpoints/latest.pt'

cmd = [
    sys.executable, 'train/pretrain.py',
    '--shards_dir',       'data/shards',
    '--ckpt_dir',         'checkpoints',
    '--log_path',         'data/train_log.csv',
    '--total_tokens',     str(1_700_000_000),
    '--batch_size',       '8',
    '--effective_tokens', str(262_144),
    '--lr',               '3e-4',
    '--min_lr',           '3e-5',
    '--warmup_frac',      '0.01',
    '--weight_decay',     '0.1',
    '--ckpt_every',       '1000',
    '--log_every',        '10',
    '--eval_every',       '500',
]
if RESUME:
    cmd += ['--resume', RESUME]

print(' '.join(cmd))
subprocess.check_call(cmd)

In [ ]:
# Cell 4 ── Monitor loss + checkpoint status
# Re-run this cell while training is in progress (in a separate kernel) to
# peek at the latest losses and saved checkpoints.
import csv, glob, os

LOG  = 'data/train_log.csv'
CKPT = 'checkpoints'

if not os.path.isfile(LOG):
    print('no log yet:', LOG)
else:
    with open(LOG, newline='', encoding='utf-8') as f:
        rows = list(csv.reader(f))
    head, body = rows[0], rows[1:]
    print('rows logged :', len(body))
    print('columns     :', head)
    print('\nlast 20 lines:')
    for r in body[-20:]:
        print('  ' + '  '.join(f'{c:>12}' for c in r))

    # Optional matplotlib plot — skip silently if matplotlib not installed.
    try:
        import matplotlib.pyplot as plt
        train_x, train_y, val_x, val_y = [], [], [], []
        for r in body:
            step = int(r[0]) if r[0] else None
            tl   = float(r[3]) if r[3] else None
            vl   = float(r[4]) if r[4] else None
            if step is None:
                continue
            if tl is not None:
                train_x.append(step); train_y.append(tl)
            if vl is not None:
                val_x.append(step); val_y.append(vl)
        plt.figure(figsize=(10, 4))
        if train_y: plt.plot(train_x, train_y, label='train', linewidth=0.7)
        if val_y:   plt.plot(val_x,   val_y,   label='val',   marker='o', linewidth=1.5)
        plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=0.3)
        plt.tight_layout(); plt.show()
    except Exception as e:
        print('(no plot:', e, ')')

ckpts = sorted(glob.glob(os.path.join(CKPT, 'step_*.pt')))
print(f'\ncheckpoints in {CKPT}: {len(ckpts)}')
for c in ckpts[-5:]:
    sz = os.path.getsize(c) / 1024**2
    print(f'  {c}  ({sz:.1f} MB)')
latest = os.path.join(CKPT, 'latest.pt')
if os.path.isfile(latest):
    print(f'  latest -> {latest}  ({os.path.getsize(latest)/1024**2:.1f} MB)')